Pixel and roi analysis

In [ ]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "seam_defect/training_config.json").is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import boto3
import pandas as pd
from pathlib import PurePosixPath

# AWS configuration
BUCKET_NAME = "amal-stitchingnet-data-903473565738-ap-southeast-2-an"
REGION = "ap-southeast-2"
DATA_PREFIX = "data/"

# Connect to S3 using SageMaker execution role
s3 = boto3.client(
    "s3",
    region_name=REGION
)

print("Connected to S3")

from seam_defect.inventory import inventory_views

(image_df, class_distribution, fabric_order, class_order, combination_counts, stratified_sample) = inventory_views(root / "reports/eda/01_s3_inventory.csv")

In [ ]:
import numpy as np
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

def load_s3_image(key):

    response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key
    )

    image = Image.open(
        BytesIO(response["Body"].read())
    ).convert("RGB")

    return image.copy()


normal_image = load_s3_image(
    "data/A. Cotton-Poly/0. Normal/A_00_001.jpg"
)

defect_image = load_s3_image(
    "data/A. Cotton-Poly/2. Broken stitch/A_02_097.jpg"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(normal_image)
axes[0].set_title("Normal Stitch")

axes[1].imshow(defect_image)
axes[1].set_title("Broken Stitch")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
normal_array = np.array(normal_image)

defect_array = np.array(defect_image)

print("Normal image:")
print("Shape:", normal_array.shape)
print("Data type:", normal_array.dtype)
print("Minimum:", normal_array.min())
print("Maximum:", normal_array.max())

print("\nDefect image:")
print("Shape:", defect_array.shape)
print("Data type:", defect_array.dtype)
print("Minimum:", defect_array.min())
print("Maximum:", defect_array.max())

In [ ]:
# Select a pixel at row 112, column 112

print(
    "Normal pixel:",
    normal_array[112, 112]
)

print(
    "Defect pixel:",
    defect_array[112, 112]
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

images = [
    ("Normal", normal_array),
    ("Broken Stitch", defect_array)
]

channels = ["Red", "Green", "Blue"]

for row, (name, array) in enumerate(images):

    for col, channel in enumerate(channels):

        axes[row, col].imshow(
            array[:, :, col],
            cmap="gray",
            vmin=0,
            vmax=255
        )

        axes[row, col].set_title(
            f"{name} - {channel}"
        )

        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, array, title in zip(
    axes,
    [normal_array, defect_array],
    ["Normal", "Broken Stitch"]
):

    for channel, color in enumerate(["red", "green", "blue"]):

        ax.hist(
            array[:, :, channel].ravel(),
            bins=256,
            range=(0, 255),
            density=True,
            histtype="step",
            color=color,
            label=color.upper()
        )

    ax.set_title(title)
    ax.set_xlabel("Pixel intensity")
    ax.set_ylabel("Relative frequency")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

def calculate_image_statistics(image):

    array = np.asarray(image).astype(np.float32)

    # Convert RGB to grayscale intensity
    grayscale = (
        0.299 * array[:, :, 0] +
        0.587 * array[:, :, 1] +
        0.114 * array[:, :, 2]
    )

    return {
        "mean_brightness": grayscale.mean(),
        "contrast_std": grayscale.std(),
        "minimum": grayscale.min(),
        "maximum": grayscale.max(),
        "median": np.median(grayscale),
        "p05": np.percentile(grayscale, 5),
        "p95": np.percentile(grayscale, 95)
    }


normal_stats = calculate_image_statistics(normal_image)

defect_stats = calculate_image_statistics(defect_image)

comparison = pd.DataFrame(
    [normal_stats, defect_stats],
    index=["Normal", "Broken Stitch"]
)

display(comparison.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

comparison["mean_brightness"].plot(
    kind="bar",
    ax=axes[0],
    title="Average Brightness"
)

comparison["contrast_std"].plot(
    kind="bar",
    ax=axes[1],
    title="Global Contrast (Standard Deviation)"
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

images = [
    ("Normal Stitch", normal_array),
    ("Broken Stitch", defect_array)
]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, (title, image) in zip(axes, images):

    ax.imshow(image)

    ax.set_title(title)

    # Show pixel coordinates
    ax.set_xticks(range(0, 225, 28))
    ax.set_yticks(range(0, 225, 28))

    ax.grid(
        True,
        color="yellow",
        alpha=0.5
    )

    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")

plt.tight_layout()
plt.show()

In [ ]:
# Candidate ROI boundaries
y_start = 80
y_end = 160

normal_roi = normal_array[
    y_start:y_end,
    :,
    :
]

defect_roi = defect_array[
    y_start:y_end,
    :,
    :
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

axes[0, 0].imshow(normal_array)
axes[0, 0].set_title("Normal - Original")

axes[0, 1].imshow(normal_roi)
axes[0, 1].set_title("Normal - Candidate ROI")

axes[1, 0].imshow(defect_array)
axes[1, 0].set_title("Broken Stitch - Original")

axes[1, 1].imshow(defect_roi)
axes[1, 1].set_title("Broken Stitch - Candidate ROI")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Define the fabric and class groups

fabric_order = sorted(
    image_df["folder_1"].unique()
)

class_order = (
    class_distribution
    .sort_values("class_id")["folder_2"]
    .tolist()
)

# Check that every combination has at least 3 images

combination_counts = (
    image_df.groupby(["folder_1", "folder_2"])
    .size()
)

print("Number of combinations:", len(combination_counts))
print("Smallest combination:", combination_counts.min())

# Select 3 random images from every combination

stratified_sample = (
    image_df
    .groupby(
        ["folder_1", "folder_2"],
        group_keys=False
    )
    .sample(
        n=3,
        random_state=42
    )
    .reset_index(drop=True)
)

print("Selected images:", len(stratified_sample))

display(
    stratified_sample[
        ["folder_1", "folder_2", "file_name"]
    ].head(15)
)

In [ ]:
from io import BytesIO
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path("reports/eda/contact_sheets")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

for class_id, class_name in enumerate(class_order):

    fig, axes = plt.subplots(
        nrows=11,
        ncols=3,
        figsize=(11, 25)
    )

    for row_idx, fabric in enumerate(fabric_order):

        group = stratified_sample[
            (stratified_sample["folder_1"] == fabric) &
            (stratified_sample["folder_2"] == class_name)
        ].sort_values("file_name")

        for col_idx, (_, record) in enumerate(
            group.iterrows()
        ):

            response = s3.get_object(
                Bucket=BUCKET_NAME,
                Key=record["s3_key"]
            )

            image = Image.open(
                BytesIO(response["Body"].read())
            ).convert("RGB")

            ax = axes[row_idx, col_idx]

            ax.imshow(image)

            ax.set_title(
                record["file_name"],
                fontsize=7
            )

            ax.axis("off")

        axes[row_idx, 0].set_ylabel(
            fabric,
            fontsize=8
        )

    fig.suptitle(
        f"Class {class_id}: {class_name}",
        fontsize=15
    )

    plt.tight_layout()

    output_path = (
        output_dir /
        f"class_{class_id:02d}.png"
    )

    plt.savefig(
        output_path,
        dpi=120,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)

    print("Saved:", output_path)

In [ ]:
from IPython.display import display, Image as NotebookImage
from pathlib import Path

output_dir = Path("reports/eda/contact_sheets")

classes_to_inspect = {
    0: "Normal",
    2: "Broken stitch",
    3: "Pinched fabric",
    6: "Puckering",
    7: "Stain and damage"
}

for class_id, class_name in classes_to_inspect.items():

    image_path = (
        output_dir / f"class_{class_id:02d}.png"
    )

    print(f"Class {class_id}: {class_name}")

    display(
        NotebookImage(filename=str(image_path))
    )

In [ ]:
from pathlib import Path
from IPython.display import display, Image as NotebookImage

output_dir = Path("reports/eda/contact_sheets")

class_names = {
    0: "Normal",
    1: "Skipped stitch",
    2: "Broken stitch",
    3: "Pinched fabric",
    4: "Crooked seam",
    5: "Thread sagging",
    6: "Puckering",
    7: "Stain and damage",
    8: "Needle mark",
    9: "Bobbin thread pulling up",
    10: "Overlapped stitch"
}

def show_contact_sheets(class_ids):

    for class_id in class_ids:

        image_path = (
            output_dir / f"class_{class_id:02d}.png"
        )

        if not image_path.exists():
            print("Missing:", image_path)
            continue

        print(
            f"Class {class_id}: "
            f"{class_names[class_id]}"
        )

        display(
            NotebookImage(
                filename=str(image_path)
            )
        )

# First batch: six classes
show_contact_sheets([1, 2, 3, 4, 5, 6])

In [ ]:
show_contact_sheets([0, 7, 8, 9])

In [ ]:

from pathlib import Path
from PIL import Image

# Local cache directory
cache_dir = output_dir / "original_cache"

cache_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Verify required columns
required_columns = [
    "file_name",
    "s3_key",
    "folder_1",
    "folder_2"
]

missing_columns = [
    col for col in required_columns
    if col not in screening_sample.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

# Prevent accidental duplicate filenames
assert screening_sample["file_name"].is_unique

downloaded_count = 0
local_paths = []

for row in screening_sample.itertuples(index=False):

    local_path = cache_dir / row.file_name

    # Download only if not already cached
    if not local_path.exists():

        s3.download_file(
            BUCKET_NAME,
            row.s3_key,
            str(local_path)
        )

        downloaded_count += 1

    # Verify image integrity
    with Image.open(local_path) as img:
        img.verify()

    local_paths.append(str(local_path))

# Store local paths in our experiment manifest
screening_sample = screening_sample.copy()

screening_sample["local_path"] = local_paths

screening_sample.to_csv(
    output_dir / "screening_121_images_with_paths.csv",
    index=False
)

print("Experiment cache completed")
print("---------------------------")
print("Total selected images:", len(screening_sample))
print("New downloads:", downloaded_count)
print("Cached images:", len(local_paths))
print("Invalid images: 0")
print("Cache directory:", cache_dir)
